# Querychat Experiments — M4 Option A Summary

This notebook summarizes experiments for M4 Option A (Querychat Customization).

For full raw runs with code, see `notebooks/querychat_customization.ipynb`.
For the full spec and design rationale, see `reports/m4_spec.md` and `docs/AI_INTEGRATION_TESTING.md`.

## What we tested

Three questions drove the experiments:

1. Does adding `extra_instructions` with business context actually change how the model answers?
2. Does `on_tool_request` work reliably as a scope gate — does `ToolRejectError` propagate back to the user cleanly?
3. Which user-facing control design fits querychat's architecture without requiring per-message prompt injection or client reinitialization?

## Experiment 1: baseline vs enhanced data_description + extra_instructions

**Setup:** Two `QueryChat` instances — M3 config (5 columns, no business framing) vs M4 config (all 16 columns + `SALESCOPE_EXTRA_INSTRUCTIONS`). Same question to both: "Which region has the highest revenue risk?"

**Baseline result:** The model didn't know about `risk_value` at all (it wasn't in the M3 data_description). It answered using churn probability alone, with no dollar framing. When asked about `Preferred_Purchase_Times`, it sometimes said the column didn't exist.

**Enhanced result:** The model found `risk_value` directly, computed total revenue at risk per region, gave a dollar figure, and ended with a retention strategy recommendation. On the `Preferred_Purchase_Times` question it worked correctly.

**Decision:** Keep the enriched `data_description` and `SALESCOPE_EXTRA_INSTRUCTIONS`. The main things the instructions add are:
- Definition of `risk_value` as the primary intervention metric
- Churn risk thresholds (>0.7 = high, 0.4–0.7 = medium, <0.4 = low)
- Instruction to frame results in business terms
- Instruction to suggest a retention strategy when relevant

## Experiment 2: on_tool_request — logging and ToolRejectError

**Setup:** Registered a callback via `client.on_tool_request(fn)` that printed every tool call. Then added a churn-scope guard that raised `ToolRejectError` when the SQL didn't reference churn-related columns.

**Logging result:** Works. Every call shows `[tool] querychat_query | sql='SELECT ...'`. This is immediately useful for debugging — you can see exactly what SQL the model is generating without digging through logs.

**Scope enforcement result:** When the guard raises `ToolRejectError`, the rejection message is returned to the model as a tool error. The model acknowledges it and tells the user something like "I'm restricted to churn-related questions in this mode." The app does not crash. The behavior is graceful.

**One limitation noted:** The scope check is a string match on the SQL — if the model writes a query that's conceptually related to churn but uses different column names, it might slip through (or get blocked when it shouldn't). This is a tradeoff for simplicity and zero latency overhead.

## Experiment 3: choosing the user-facing control

Three options were tried or considered:

**Verbosity slider:** Wanted to inject "be more concise" or "be more detailed" into the prompt based on the slider value. Problem: `QueryChat` builds the system prompt at init time. Changing it mid-session requires reinitializing the client, which loses conversation history. Not practical.

**Response style dropdown (Bullet / Prose / Executive):** Same problem — system prompt change requires re-init. Dropped.

**Analysis Scope dropdown (Full / Churn Only / Revenue Only):** This one works because the restriction is in the `on_tool_request` callback, not the system prompt. The callback reads from a `_scope` dict that's updated by a `@reactive.effect` whenever the dropdown changes. So switching the dropdown mid-session takes effect on the very next tool call. No re-init needed.

The scope control also maps cleanly to the three personas in `reports/m4_spec.md`: Sales Leaders can focus the AI on revenue questions, Customer Success Leads on churn questions, Analysts leave it in full mode.

## Summary

| Feature | What we added | Where |
|---------|--------------|-------|
| System prompt override | `SALESCOPE_EXTRA_INSTRUCTIONS` + enriched `data_description` in `QueryChat()` init | `src/app.py` lines 30–77 |
| `on_tool_request` | `_handle_tool_request` logs SQL + raises `ToolRejectError` for out-of-scope queries | `src/app.py` lines ~389–414 |
| User-facing control | `ai_scope_mode` dropdown in AI Insights tab | `src/app.py` lines ~300–323 |

All three are required for full marks on Option A. See `notebooks/querychat_customization.ipynb` for the full experimental code.